# Quantum Challenge: Entanglement as a Computing Resource [100 points]
**Level:** Beginner • **Duration:** 60–90 minutes • **Platform:** Local Qiskit + Aer

## Mission
You are building a miniature quantum communication system. First learn to control qubits, then create and test an entangled pair, and finally **use that pair to teleport a quantum state**. Teleportation is the final demonstration that entanglement is a *resource*, not merely an unusual correlation.

**Rules:** Complete each `TODO` cell. Run its check immediately afterward. Do not edit the checks. All circuits run on your own computer; no IBM account, cloud service, hardware, or device emulator is needed.

**Learning objectives:** use X/H/Z/Ry gates; apply CX/CZ; select qubit indices correctly; measure and interpret counts; distinguish correlations from entanglement; use a Bell pair in a teleportation circuit.

**Scoring:** 5 tasks × 20 points = 100 points. Reflection questions are for discussion.

## Setup
Install packages once if needed (uncomment the next line), then restart the kernel. Requires a modern Qiskit installation.

In [27]:
# %pip install qiskit qiskit-aer numpy matplotlib

In [28]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity
from qiskit_aer import AerSimulator

simulator = AerSimulator(seed_simulator=42)
SHOTS = 2048

def sample_counts(circuit, shots=SHOTS):
    return simulator.run(circuit, shots=shots).result().get_counts()

def show(circuit):
    display(circuit.draw(output="text"))

print("Ready: local Aer simulator")

Ready: local Aer simulator


## Task 1 — Control one qubit (20 points)
Start in $|0\rangle$. Apply **X** to flip it to $|1\rangle$, then **H** to create a superposition. In a separate circuit, use **Ry(π/3)** to prepare a tunable state.

**Your job:** fill in all three gate operations. Remember: `qc.x(0)` acts on qubit 0.

**Predict:** What probabilities should you see after X then H?

In [29]:
single = QuantumCircuit(1)
single.x(0)
single.h(0)

rotated = QuantumCircuit(1)
rotated.ry(np.pi/3, 0)

show(single)
show(rotated)

┌───┐┌───┐
q: ┤ X ├┤ H ├
   └───┘└───┘

┌─────────┐
q: ┤ Ry(π/3) ├
   └─────────┘

In [30]:
# CHECK 1
expected_single = QuantumCircuit(1); expected_single.x(0); expected_single.h(0)
expected_rot = QuantumCircuit(1); expected_rot.ry(np.pi/3, 0)
assert Statevector.from_instruction(single).equiv(Statevector.from_instruction(expected_single)), "Check X then H."
assert Statevector.from_instruction(rotated).equiv(Statevector.from_instruction(expected_rot)), "Check Ry angle and target."
print("Task 1 passed! +20 points")

Task 1 passed! +20 points


## Task 2 — Build your entanglement resource (20 points)
Make the Bell state
$$|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt{2}.$$

**Your job:** apply H to qubit 0 and then CX with **control 0, target 1**. Why can't this state be written as a product of two independent single-qubit states?

In [31]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
show(bell)

┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

In [32]:
# CHECK 2
expected_bell = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
assert Statevector.from_instruction(bell).equiv(expected_bell), "Expected |Phi+>."
# Each qubit alone is maximally mixed: evidence of entanglement for this pure 2-qubit state.
rho0 = partial_trace(Statevector.from_instruction(bell), [1])
assert np.allclose(rho0.data, np.eye(2)/2), "Reduced state should be I/2."
print("Task 2 passed! +20 points")

Task 2 passed! +20 points


## Task 3 — Measure correlations (20 points)
An entangled Bell pair measured in the Z basis gives only `00` and `11` (roughly half each). A *classical mixture* can show the same Z-basis counts, so **matching outcomes alone does not prove entanglement**.

**Your job:** make a measured copy of your Bell circuit, run 2048 shots, and report the counts. Then measure in the **X basis** by applying H to both qubits before measurement. The Bell state also has matching X-basis outcomes.

Qiskit count strings are displayed with the highest-index classical bit on the left.

In [33]:
measured_z = bell.copy()
measured_z.measure_all()

measured_x = bell.copy()
measured_x.h(0)
measured_x.h(1)
measured_x.measure_all()

print("Z basis:", sample_counts(measured_z))
print("X basis:", sample_counts(measured_x))

Z basis: {'11': 1047, '00': 1001}
X basis: {'11': 1047, '00': 1001}


In [34]:
# CHECK 3
assert measured_z.num_clbits == 2 and measured_x.num_clbits == 2, "Measure both qubits."
cz = sample_counts(measured_z)
cx = sample_counts(measured_x)
assert set(cz).issubset({"00", "11"}), "Z outcomes must match."
assert set(cx).issubset({"00", "11"}), "X outcomes must match."
assert all(0.4 < c.get(k, 0)/SHOTS < 0.6 for c in (cz, cx) for k in ("00", "11")), "Expect near 50/50 counts."
print("Z:", cz, " X:", cx)
print("Task 3 passed! +20 points")

Z: {'11': 1047, '00': 1001}  X: {'11': 1047, '00': 1001}
Task 3 passed! +20 points


## Task 4 — Change the entangled state (20 points)
Start from $|\Phi^+\rangle$. Apply **Z to qubit 1** to create
$$|\Phi^-\rangle = (|00\rangle-|11\rangle)/\sqrt{2}.$$

The probabilities in the Z basis are unchanged, but the **relative phase** changes the X-basis correlations. Then apply Z again to undo the change.

**Your job:** manipulate qubit 1 and verify both states. Discuss: why can't Z-basis counts reveal the relative minus sign?

In [35]:
phase_flip = bell.copy()
phase_flip.z(1)

undo_phase = phase_flip.copy()
undo_phase.z(1)

show(phase_flip)

┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐┌───┐
q_1: ─────┤ X ├┤ Z ├
          └───┘└───┘

In [36]:
# CHECK 4
phi_minus = Statevector([1/np.sqrt(2), 0, 0, -1/np.sqrt(2)])
assert Statevector.from_instruction(phase_flip).equiv(phi_minus), "Need |Phi->."
assert Statevector.from_instruction(undo_phase).equiv(expected_bell), "Z twice should undo."
print("Task 4 passed! +20 points")

Task 4 passed! +20 points


## Task 5 — Final mission: teleport a quantum state (20 points)
**Resource:** qubits 1 and 2 share an entangled Bell pair. Qubit 0 holds an input state $|\psi\rangle = R_y(\pi/3)|0\rangle$ that Alice wants Bob to receive.

The standard teleportation protocol measures Alice's qubits and sends **two classical bits** to Bob, who applies corrections. For a simple statevector check, we use the **deferred-measurement principle**: replace those classically controlled corrections with coherent controlled gates. This version has the same final state on Bob's qubit, but it does *not* demonstrate a physical classical-message exchange.

**Your job:** fill in the six gate operations below, with these qubit roles:

1. Entanglement resource: H(1), CX(1,2).
2. Alice's Bell-basis interaction: CX(0,1), H(0).
3. Bob's coherent corrections: CX(1,2), CZ(0,2).

**Important:** A Bell pair is consumed in the protocol; the input is transferred to qubit 2 rather than copied. Quantum teleportation does **not** allow faster-than-light messaging.

In [37]:
teleport = QuantumCircuit(3)
teleport.ry(np.pi/3, 0)  # Alice's unknown-to-Bob input state
teleport.h(1)
teleport.cx(1, 2)
teleport.cx(0, 1)
teleport.h(0)
teleport.cx(1, 2)
teleport.cz(0, 2)
show(teleport)

┌─────────┐          ┌───┐   
q_0: ┤ Ry(π/3) ├───────■──┤ H ├─■─
     └──┬───┬──┘     ┌─┴─┐└───┘ │ 
q_1: ───┤ H ├─────■──┤ X ├──■───┼─
        └───┘   ┌─┴─┐└───┘┌─┴─┐ │ 
q_2: ───────────┤ X ├─────┤ X ├─■─
                └───┘     └───┘

In [38]:
# CHECK 5: Bob's reduced density matrix should equal the input state.
input_circuit = QuantumCircuit(1)
input_circuit.ry(np.pi/3, 0)
input_state = Statevector.from_instruction(input_circuit)
final_state = Statevector.from_instruction(teleport)
bob = partial_trace(final_state, [0, 1])
fidelity = state_fidelity(bob, DensityMatrix(input_state))
print(f"Bob's fidelity: {fidelity:.6f}")
assert fidelity > 1 - 1e-10, "Teleportation failed: check gate order and qubit indices."
# Verify that the source is not left as an identical independent copy.
alice_input_qubit = partial_trace(final_state, [1, 2])
assert state_fidelity(alice_input_qubit, DensityMatrix(input_state)) < 0.999, "Check the teleportation circuit."
print("Task 5 passed! +20 points")

Bob's fidelity: 1.000000
Task 5 passed! +20 points


## Reflection and discussion
Answer in a new Markdown cell:

1. Which gate **creates superposition**, and which two-qubit gate **creates entanglement** in Task 2?
2. Why do perfect Z-basis correlations not, by themselves, prove entanglement?
3. What changes when Z is applied to one qubit of a Bell pair?
4. In Task 5, which qubits are Alice's and which is Bob's?
5. What resource is consumed by teleportation? Why are classical communication and corrections required in the usual measured protocol?

### Optional extension (no extra points)
Replace the coherent corrections in Task 5 with mid-circuit measurements and conditional X/Z corrections using Qiskit's `if_test`, then compare Bob's measurement statistics across different input states. This extension may require attention to the installed Qiskit/Aer version.

**Instructor note:** The assertions are formative local checks, not a secure autograder. Students can run the notebook entirely offline after package installation.

## Reflection answers

1. **H** creates the superposition (qubit 0 becomes |+>); **CX** (control 0, target 1) creates the entanglement by copying the "which branch" information onto qubit 1.
2. Matching Z-basis outcomes (only 00 and 11) can also come from a classical mixture of |00> and |11>, e.g. two coins that were prepared equal. Entanglement also needs correlations in other bases (here X-basis) and the coherent phase between the branches, which a Z-basis measurement cannot see.
3. Z on one qubit turns |Phi+> into |Phi->: the relative phase between |00> and |11> flips sign. Z-basis probabilities stay 50/50, but X-basis correlations become anti-correlated (outcomes 01/10 instead of 00/11). Applying Z again restores |Phi+>.
4. Alice holds qubit 0 (the input state) and qubit 1 (her half of the Bell pair). Bob holds qubit 2.
5. The shared Bell pair (entanglement) is consumed. In the usual protocol Alice's measurement outcomes are random; only after she sends the 2 classical bits can Bob apply the right X/Z correction. Without them Bob's qubit is maximally mixed, which is why teleportation cannot send information faster than light.
